# Web Scraping Pipeline - La Jornada Maya 

## Project Overview
This notebook demonstrates the extraction of structured metadata and news articles from the Mayan-language section (*K'iintsil*) of **La Jornada Maya** for the year 2021. 

In [11]:
import os
import re
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup
from fake_useragent import UserAgent

# Resolve path to the data directory from the notebooks folder
BASE_DIR = Path(os.getcwd()).parent
DATA_DIR = BASE_DIR / "data"

print(f"Project Base Directory: {BASE_DIR}")
print(f"Target Data Directory: {DATA_DIR}")

Project Base Directory: c:\Users\maric\Downloads\WebScraping_JornadaMaya
Target Data Directory: c:\Users\maric\Downloads\WebScraping_JornadaMaya\data


### 1. URL Discovery & Link Extraction
Implements the ingestion layer by connecting to the archive catalog pages. It injects a randomized `UserAgent` header to mitigate rate-limiting and anti-bot blocks, parsing individual article reference handles for the extraction pipeline.

In [12]:
def get_links(url):
    """Fetches all article links from a given category overview page"""
    try:
        ua = UserAgent()
        headers = {"User-Agent" : ua.random}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "html5lib")
        all_links = []
        
        contents = soup.find_all("div", class_="single-blog-content")
        for content in contents:
            link = content.find("a", target = "_blank")
            if link and link.get("href"):
                all_links.append(link.get("href"))
        
        return all_links
    
    except requests.exceptions.RequestException as e:
        print(f"[Error] Failed to fetch URL {url}: {e}")
        return []
    
def find_all_links_in_month(month):
    """Constructs the URL for a specific month and retrieves its links"""
    url = f"https://www.lajornadamaya.mx/k'iintsil/{month}-2021"
    return get_links(url)

### 2. Document Parsing & Metadata Extraction
Handles document parsing by downloading each standalone article layout. The `get_info` function combines precise CSS selectors with Regular Expressions (Regex) to isolate structural features (e.g., translation attributes and publishing origins) with robust exception handling.

In [14]:

def get_info(link):
    """Scrapes structured metadata and content from a single article link"""
    try:
        ua = UserAgent()
        headers = {"User-Agent" : ua.random}
        response = requests.get(link, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "html5lib")
        
        # Extract Title
        try:
            title = soup.find("h1").get_text(strip=True) 
        except AttributeError:
            print(f"[Warning] Title not found: {link}")
            title = "Untitled"
            
        # Extract Date
        try:
            info = soup.find("div", class_="post-meta")
            if not info:
                raise ValueError("IPost metadata section not found")
            
            date_pattern = r"\d{2}/\d{2}/\d{4}"
            date_match = re.search(date_pattern, info.get_text())
            if not date_match:
                raise ValueError("Date pattern matching failed")
             
            date = date_match.group().strip() 
            
        except (AttributeError, ValueError) as e:
            print(f"[Warning] Error extracting data from {link} : {e}")
            date = "Unknown Date"
            
        # Extract Location
        try: 
            location = info.find_next("p").find_next("p").get_text(strip=True)[13:]
            if not location:
                location = "Unknown Location"
        except AttributeError:
            print(f"[Warning] Location not found: {link}")
            location = "Unknown Location"
            
        # Extract Article Body Content
        try:
            article = info.find_next("p").find_next("p")
            text = ""
            if not article:
                raise ValueError("Article body root element not found")
            
            paragraphs = article.find_all_next("p")
            for paragraph in paragraphs :
                if not paragraph.find("strong"):
                    text += paragraph.get_text() + "\n"
                
        except (AttributeError, ValueError) as e:
            print(f"[Warning] Error extracting article body from {link} : {e}")
            text = "Content Unavailable"
        
        # Extract Author
        try:
            author = info.find_next("p").get_text()
            if not author:
                author = "Unknown Author"
        except AttributeError:
            print(f"[Warning] Error extracting author from {link}")
            author = "Unknown Author"
        
        # Extract Maya Translator
        try:     
            content = soup.find_all("div", class_="single-blog-content")[-1]    
            translator_pattern = r"Xak'alutskíinsa'an tumen:\s+(\w+\s\w+\s\w+)"
            translator_match = re.search( translator_pattern, content.get_text())
            if translator_match:
                translator = translator_match.group(1).strip()
            else:
                translator = "Unknown Translator"
                
        except (IndexError, AttributeError):
            print(f"[Warning] Error extracting tranaltor from {link}")
            translator = "Unknown Translator"
        
        return {
            "Date": date,
            # "Autor": author,
            # "Translator": translator,
            # "Location": location,
            "Title": title,
            "Content": text.strip()
        }
         
    except requests.exceptions.RequestException as e:
        print(f"[Error] Network failure on URL {link}: {e}")
        return None

### 3. Pipeline Execution Loop
The core execution loop of the ETL pipeline. It orchestrates the crawler across target timeframes, collect payloads into structured dictionaries, pipelines the data ecosystem into **Pandas**, and exports a persistent dataset.

In [ ]:
"""Main execution loop to pipeline scraping processes into CSV output"""
    
# months = [
#    "enero", "febrero", "marzo", "abril", "mayo", "junio", 
#    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
#]
# To test quickly inside the notebook, we can isolate to one month first
months = ["enero"]
dfs = []
    
for month in months:
    all_links = find_all_links_in_month(month)
    data_month = []

    print(f"Processing Month:{month} | Total links found: {len(all_links)} \n")
        
    for link in all_links:
         if link: 
            info = get_info(link)
            if info:
                data_month.append(info)
        
    if data_month:
        df_month = pd.DataFrame(data_month)
        dfs.append(df_month)
            
if dfs:
    final_df = pd.concat(dfs, ignore_index=True)
    output_file = DATA_DIR / "processed_articles_enero.csv"
    final_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"[Success] Scraping execution completed. Dataset saved to {output_file}")
else: 
    print("[Info] No data extracted across the execution target.")

Processing Month:enero | Total links found: 55 

[Success] Scraping execution completed. Dataset saved to c:\Users\maric\Downloads\WebScraping_JornadaMaya\data\datos_enero.csv


### 4. Dataset Evaluation
Performs data validation and schema verification. This step inspects the top rows of the engineered tabular dataset to confirm that data types, encoding formats, and features are optimal before downstream usage.

In [16]:
# Read back the saved file to verify completeness
if output_file.exists():
    preview_df = pd.read_csv(output_file)
    display(preview_df.head())
else:
    print("Execute the pipeline step above to generate data previews.")

,Date,Title,Content
0,04/01/2021,Ribereñoil jchuk kayo’obe’ táanil yano'ob ichi...,Juan Carlos Pérez\nU nu’ukulo’obe’ laj laabtak...
1,04/01/2021,Oaxacae’ tu chíimpoltaj jmeyaj k’ab úuchik u c...,Te’e k’iino’oba’ u mola’ayil Instituto Oaxaque...
2,05/01/2021,"Xok k’iin, úuchben kaambal láayli’ u beeta’al ...","Xok k’iine’, leti’ bix u na’atal u máan k’iin ..."
3,05/01/2021,U íinjenieroilo’ob Googlee’ tu káajsajo’ob u m...,Junmúuch’ u íinjenieroilo’ob Google’ tu k’a’ay...
4,05/01/2021,"Hazarao’ob kíinsa’ab Pakistáne’, bíin mu’ukuko...",U láak’o’ob buluktúul hazara wíinik kíinsa’ab ...
